In this notebook, I want mainly want to try to feature engineer up to degree two and see if interactions between stats have any more predictive power. The thing is, this level of dimensionality (around 3000 features) will cause extreme overfitting of the data in a non-regularized logistic regressor, but forward feature selection to select features is difficult due to extreme computational requirements when the degree gets that high. So, I will try to use lasso regression to do implicit feature selection on this higher dimension dataset to try to reduce some dimensionality more feasibly. I will not go beyond degree two since degree three already jumps from ~3000 features to ~79000 features which is computationally infeasible.

In [2]:
import pandas as pd
df = pd.read_csv("../../data/aggregated_rolling_features.csv")
df.shape

(5539, 97)

In [3]:
to_drop = []
to_drop.extend([f"team{i}_player{k}_id" for i in range(1,3) for k in range(1,6)])
to_drop.extend([f"team{i}_id" for i in range(1,3)])
to_drop.extend([f"team{i}" for i in range(1,3)])
to_drop.extend(["tournament", "match_id", "game_id", "map_id", "map_name", "datetime", "Unnamed: 0"])
df = df.drop(to_drop, axis=1)
df

,team1_win,bestOf,team1_previous_10_average_map_score,team2_previous_10_average_map_score,previous_10_games_team1_average_kills,previous_10_games_team1_std_kills,previous_10_games_team1_range_kills,previous_10_games_team1_max_kills,previous_10_games_team1_min_kills,previous_10_games_team1_median_kills,...,previous_10_games_team2_range_kast,previous_10_games_team2_max_kast,previous_10_games_team2_min_kast,previous_10_games_team2_median_kast,previous_10_games_team2_average_kddiff,previous_10_games_team2_std_kddiff,previous_10_games_team2_range_kddiff,previous_10_games_team2_max_kddiff,previous_10_games_team2_min_kddiff,previous_10_games_team2_median_kddiff
0,0,3.0,13.000000,13.000000,16.000000,0.000000e+00,0.0,16.000000,16.000000,16.000000,...,0.00,70.000000,70.000000,70.000000,1.000000,0.000000,0.0,1.000000,1.000000,1.000000
1,1,3.0,12.333333,12.333333,15.600000,4.127953e+00,10.0,22.000000,12.000000,13.000000,...,25.00,83.300000,58.300000,58.300000,-0.600000,3.136877,9.0,4.000000,-5.000000,0.000000
2,1,3.0,11.600000,11.600000,15.100000,2.083267e+00,5.5,18.500000,13.000000,14.000000,...,17.30,75.000000,57.700000,60.100000,-1.600000,5.885576,17.5,8.500000,-9.000000,-1.500000
3,1,3.0,11.571429,11.571429,14.612903,1.776357e-15,0.0,14.612903,14.612903,14.612903,...,0.00,71.535484,71.535484,71.535484,-0.032258,0.000000,0.0,-0.032258,-0.032258,-0.032258
4,1,3.0,10.666667,10.666667,13.600000,4.363485e+00,13.0,21.000000,8.000000,13.000000,...,33.30,60.000000,26.700000,33.300000,-9.000000,2.280351,6.0,-6.000000,-12.000000,-9.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5534,0,3.0,13.400000,9.500000,14.140000,9.200000e-01,2.6,15.400000,12.800000,14.200000,...,6.89,77.400000,70.510000,76.760000,0.780000,2.465279,6.4,3.500000,-2.900000,1.800000
5535,0,3.0,9.700000,11.100000,14.680000,1.175415e+00,3.3,16.600000,13.300000,14.900000,...,6.96,78.300000,71.340000,76.820000,0.920000,2.741095,7.0,3.700000,-3.300000,2.300000
5536,1,3.0,11.000000,10.700000,14.440000,6.590903e-01,1.7,15.400000,13.700000,14.600000,...,6.96,79.800000,72.840000,75.820000,1.140000,2.793278,7.1,3.900000,-3.200000,2.700000
5537,0,5.0,11.400000,12.700000,14.780000,2.594147e+00,6.8,17.500000,10.700000,14.900000,...,5.71,83.730000,78.020000,82.000000,4.280000,3.398470,8.7,7.800000,-0.900000,5.200000


In [4]:
X = df.drop("team1_win", axis=1)
y = df.team1_win

In [5]:
X_numerical = X.drop("bestOf", axis=1)
X_categorical = X.bestOf
X_categorical = pd.get_dummies(X_categorical, drop_first=True,dtype=int,prefix="bestOf")

Let's get the feature engineered set before standardizing

In [6]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2,include_bias=False)
X_numerical_poly_prime = poly.fit_transform(X_numerical)
X_numerical_poly = pd.DataFrame(X_numerical_poly_prime, columns=poly.get_feature_names_out(X_numerical.columns))
X_poly = pd.concat([X_numerical_poly, X_categorical], axis=1)
X_poly.shape

(5539, 2851)

Standardize

In [7]:
from sklearn.preprocessing import StandardScaler
from math import ceil
stnd = StandardScaler().set_output(transform="pandas")
#Split
split_point = ceil(len(df) * 0.8)
train_numerical = X_numerical.iloc[:split_point] # 0 to split_point - 1
test_numerical = X_numerical.iloc[split_point:] # split_point to len(df)
train_poly = X_poly.iloc[:split_point] # 0 to split_point - 1
test_poly = X_poly.iloc[split_point:] # split_point to len(df)
train_cat = X_categorical.iloc[:split_point]
test_cat = X_categorical.iloc[split_point:]
y_train = y.iloc[:split_point]
y_test = y.iloc[split_point:]
#Standardize
train_numerical = stnd.fit_transform(train_numerical)
test_numerical = stnd.transform(test_numerical)
train_poly = stnd.fit_transform(train_poly)
test_poly = stnd.transform(test_poly)
#Concat
X_train = pd.concat([train_numerical, train_cat], axis=1)
X_test = pd.concat([test_numerical, test_cat], axis=1)
X_poly_train = pd.concat([train_poly, train_cat], axis=1)
X_poly_test = pd.concat([test_poly, test_cat], axis=1)
#Show results
print("X_train shape", X_train.shape)
print("X_test shape", X_test.shape)
print("X_poly_train shape", X_poly_train.shape)
print("X_poly_test shape", X_poly_test.shape)

X_train shape (4432, 76)
X_test shape (1107, 76)
X_poly_train shape (4432, 2853)
X_poly_test shape (1107, 2853)


# First, let's get a baseline view of how lasso performs before we feature engineer

First, grid search for ideal alpha value

In [8]:
import numpy as np
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression

grid = {"C": np.logspace(-5, 5, num=11)}
#Asked ChatGPT why I was getting a solver error
# liblinear or saga support l1; use liblinear for binary problems
lr_est = LogisticRegression(l1_ratio=1, solver="liblinear", max_iter=200, verbose=2)
cv = GridSearchCV(lr_est, param_grid=grid, n_jobs=-1, cv=TimeSeriesSplit(), verbose=2)
cv.fit(X_train, y_train)

Fitting 5 folds for each of 11 candidates, totalling 55 fits
[LibLinear][LibLinear][LibLinear]=========================
optimization finished, #iter = 0
Objective value = 0.102586
#nonzeros/#features = 0/77
[LibLinear]=========================
optimization finished, #iter = 0
Objective value = 0.015374
#nonzeros/#features = 0/77
[CV] END ...........................................C=0.0001; total time=   0.0s
optimization finished, #iter = 0
Objective value = 0.204894
#nonzeros/#features = 0/77
[CV] END ............................................C=1e-05; total time=   0.0s
optimization finished, #iter = 0
Objective value = 2.560486
#nonzeros/#features = 0/77
[CV] END ...........................................C=0.0001; total time=   0.0s
[CV] END ............................................C=0.001; total time=   0.0s
[LibLinear]=========================
optimization finished, #iter = 0
Objective value = 0.051432
#nonzeros/#features = 0/77
[CV] END ......................................

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LogisticRegre...r', verbose=2)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': array([1.e-05...e+04, 1.e+05])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displa

The model chose 26 nonzero features.

Let's see which C was best (inverse of alpha)

In [9]:
cv.best_params_

{'C': np.float64(0.1)}

So C of 0.1 or alpha of 10 is best.

In [10]:
best_estimator = cv.best_estimator_
best_estimator.score(X_train, y_train)

0.5943140794223827

In [11]:
best_estimator.score(X_test, y_test)

0.5709123757904245

It is able to score ~2% better than baseline on the test set.

How did lasso distribute weights?

In [12]:
weights = pd.DataFrame()
weights["Feature"] = X_train.columns
weights["Weight"] = best_estimator.coef_[0]
weights = weights.sort_values(by="Weight", ascending = False)
display(weights.head(10))
display(weights.tail(10))

,Feature,Weight
5,previous_10_games_team1_max_kills,0.123737
31,previous_10_games_team1_median_kast,0.085156
30,previous_10_games_team1_min_kast,0.074166
55,previous_10_games_team2_median_assists,0.070546
0,team1_previous_10_average_map_score,0.052362
24,previous_10_games_team1_min_adr,0.037635
63,previous_10_games_team2_std_kast,0.037346
15,previous_10_games_team1_std_assists,0.026023
46,previous_10_games_team2_range_deaths,0.023498
33,previous_10_games_team1_std_kddiff,0.015203


,Feature,Weight
7,previous_10_games_team1_median_kills,-0.015606
62,previous_10_games_team2_average_kast,-0.019173
69,previous_10_games_team2_std_kddiff,-0.021030
18,previous_10_games_team1_min_assists,-0.027297
11,previous_10_games_team1_max_deaths,-0.062317
9,previous_10_games_team1_std_deaths,-0.077430
60,previous_10_games_team2_min_adr,-0.078224
41,previous_10_games_team2_max_kills,-0.089118
1,team2_previous_10_average_map_score,-0.110423
73,previous_10_games_team2_median_kddiff,-0.110749


The model seems to find team2 previous median kddiff, team2 previous average map score, and team 1 previous max kills the most important, which is different from what the collinear feature-pruned model valued. The collinear feature-pruned logistic regressor performed slightly better by ~0.3%.

# Now, let's try doing the feature engineered set.

In [13]:
cv = GridSearchCV(lr_est, param_grid=grid, n_jobs=-1, cv=TimeSeriesSplit(), verbose=2)
cv.fit(X_poly_train, y_train)

Fitting 5 folds for each of 11 candidates, totalling 55 fits
[LibLinear]=========================
optimization finished, #iter = 0
Objective value = 0.005143
#nonzeros/#features = 0/2854
[LibLinear][CV] END ............................................C=1e-05; total time=   0.1s
[LibLinear][LibLinear][LibLinear][LibLinear][LibLinear]=========================
optimization finished, #iter = 0
Objective value = 0.051432
#nonzeros/#features = 0/2854
[LibLinear][CV] END ...........................................C=0.0001; total time=   0.1s
[LibLinear][LibLinear][LibLinear]=========================
optimization finished, #iter = 0
Objective value = 0.010259
#nonzeros/#features = 0/2854
[CV] END ............................................C=1e-05; total time=   0.2s
optimization finished, #iter = 0
Objective value = 0.025605
#nonzeros/#features = 0/2854
[LibLinear]=========================
optimization finished, #iter = 0
Objective value = 0.020489
#nonzeros/#features = 0/2854
[LibLinear][Lib

/home/hixonbm/cssema415/src/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


iter 103  #CD cycles 1000
iter  81  #CD cycles 1000
iter 139  #CD cycles 1000
iter  99  #CD cycles 1000
iter 187  #CD cycles 1000
iter 146  #CD cycles 1000
iter 105  #CD cycles 1000
iter 131  #CD cycles 1000
iter  93  #CD cycles 1000
iter  81  #CD cycles 1000
iter 140  #CD cycles 1000
iter 188  #CD cycles 1000
iter 147  #CD cycles 1000
iter 104  #CD cycles 1000
iter  92  #CD cycles 1000
iter  82  #CD cycles 1000
iter 100  #CD cycles 1000
iter 132  #CD cycles 1000
iter 106  #CD cycles 1000
iter 141  #CD cycles 1000
iter 189  #CD cycles 1000
iter  94  #CD cycles 1000
iter 148  #CD cycles 1000
iter  82  #CD cycles 1000
iter 105  #CD cycles 1000
iter  93  #CD cycles 1000
iter 133  #CD cycles 1000
iter 142  #CD cycles 1000
iter 101  #CD cycles 1000
iter  83  #CD cycles 1000
iter 190  #CD cycles 1000
iter 107  #CD cycles 1000
iter 149  #CD cycles 1000
iter  95  #CD cycles 1000
iter 106  #CD cycles 1000
iter 134  #CD cycles 1000
iter 143  #CD cycles 1000
iter  83  #CD cycles 1000
iter 191  #C

/home/hixonbm/cssema415/src/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


iter 152  #CD cycles 1000
iter 159  #CD cycles 1000
iter 108  #CD cycles 1000
iter 115  #CD cycles 1000
iter 103  #CD cycles 1000
iter  89  #CD cycles 1000
iter 153  #CD cycles 1000
iter 160  #CD cycles 1000
iter 100  #CD cycles 1000
iter 114  #CD cycles 1000
iter  90  #CD cycles 1000
iter 143  #CD cycles 1000
iter 116  #CD cycles 1000
iter 109  #CD cycles 1000
iter 115  #CD cycles 586
iter 104  #CD cycles 1000
iter 116  #CD cycles 57
iter 154  #CD cycles 1000
iter 161  #CD cycles 1000
iter 117  #CD cycles 142
iter 118  #CD cycles 1
iter  90  #CD cycles 1000
iter 144  #CD cycles 1000
iter 101  #CD cycles 1000
iter  91  #CD cycles 1000
iter 155  #CD cycles 1000
iter 162  #CD cycles 1000
iter 117  #CD cycles 1000
iter 105  #CD cycles 1000
iter 110  #CD cycles 1000
iter 119  #CD cycles 1000
iter 145  #CD cycles 1000
iter 156  #CD cycles 1000
iter  91  #CD cycles 1000
iter 163  #CD cycles 1000
iter 102  #CD cycles 1000
iter  92  #CD cycles 1000
iter 118  #CD cycles 1000
iter 106  #CD cycle

...

/home/hixonbm/cssema415/src/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


iter 185  #CD cycles 1000
iter 186  #CD cycles 1000
iter 187  #CD cycles 1000
iter 188  #CD cycles 1000
iter 189  #CD cycles 1000
iter 190  #CD cycles 1000
iter 191  #CD cycles 1000
iter 192  #CD cycles 1000
iter 193  #CD cycles 1000
iter 194  #CD cycles 1000
iter 195  #CD cycles 1000
iter 196  #CD cycles 1000
iter 197  #CD cycles 1000
iter 198  #CD cycles 1000
iter 199  #CD cycles 1000
iter 200  #CD cycles 1000
optimization finished, #iter = 200
Objective value = 12292678.241247
#nonzeros/#features = 2853/2854
[CV] END ..........................................C=10000.0; total time=68.5min


/home/hixonbm/cssema415/src/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:1258: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


[LibLinear]iter   1  #CD cycles 1
iter   2  #CD cycles 1
iter   3  #CD cycles 1
iter   4  #CD cycles 1
iter   5  #CD cycles 3
iter   6  #CD cycles 1
iter   7  #CD cycles 8
iter   8  #CD cycles 1
iter   9  #CD cycles 17
iter  10  #CD cycles 1
iter  11  #CD cycles 68
iter  12  #CD cycles 1
optimization finished, #iter = 12
Objective value = 296.053268
#nonzeros/#features = 97/2854


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LogisticRegre...r', verbose=2)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': array([1.e-05...e+04, 1.e+05])}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",TimeSeriesSpl...est_size=None)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displa

The model chose 97 nonzero features.

In [14]:
cv.best_params_

{'C': np.float64(0.1)}

The ideal C is 0.1 and ideal alpha is 10.

In [15]:
best_estimator = cv.best_estimator_

In [ ]:
weights = pd.DataFrame()
weights["Feature"] = X_poly_train.columns
weights["Weight"] = best_estimator.coef_[0]
weights = weights.sort_values(by="Weight", ascending = False)
display(weights.head(10))
display(weights.tail(10))

,Feature,Weight
1772,previous_10_games_team1_range_kast previous_10...,0.141591
1495,previous_10_games_team1_range_adr previous_10_...,0.114983
2470,previous_10_games_team2_range_deaths previous_...,0.105773
1193,previous_10_games_team1_range_assists previous...,0.097866
1267,previous_10_games_team1_min_assists previous_1...,0.096495
1082,previous_10_games_team1_std_assists previous_1...,0.092180
2145,previous_10_games_team1_min_kddiff previous_10...,0.090330
414,previous_10_games_team1_range_kills previous_1...,0.071848
1960,previous_10_games_team1_average_kddiff previou...,0.064902
2657,previous_10_games_team2_min_assists previous_1...,0.062628


,Feature,Weight
216,team2_previous_10_average_map_score previous_1...,-0.062417
2169,previous_10_games_team1_median_kddiff previous...,-0.065599
2161,previous_10_games_team1_median_kddiff previous...,-0.080650
2069,previous_10_games_team1_max_kddiff^2,-0.081494
624,previous_10_games_team1_median_kills previous_...,-0.091414
164,team2_previous_10_average_map_score previous_1...,-0.110048
2802,previous_10_games_team2_range_kast previous_10...,-0.117023
2841,previous_10_games_team2_range_kddiff previous_...,-0.119625
1137,previous_10_games_team1_std_assists previous_1...,-0.153820
1461,previous_10_games_team1_std_adr previous_10_ga...,-0.209154


In [20]:
with pd.option_context('display.max_colwidth', None):
    print(weights.Feature.head(1))
    print(weights.Feature.tail(3))

1772    previous_10_games_team1_range_kast previous_10_games_team1_average_kddiff
Name: Feature, dtype: object
2841      previous_10_games_team2_range_kddiff previous_10_games_team2_min_kddiff
1137    previous_10_games_team1_std_assists previous_10_games_team2_median_kddiff
1461           previous_10_games_team1_std_adr previous_10_games_team2_range_kast
Name: Feature, dtype: object


Looks like the 4 most important features according to the model, all being interaction terms, are team 2 range kddiff and team 2 min kddiff; team 1 std assists and team 2 median kddiff; team 1 std adr and team 2 range kast; and team 1's kast range and team 1's average kddiff.

In [21]:
best_estimator.score(X_poly_train, y_train)

0.6042418772563177

In [22]:
best_estimator.score(X_poly_test, y_test)

0.5573622402890696

It does much worse than the original L1 regularized feature set. I think the reason these features do well in the train test and validation tests is because at the time training data is collected and analyzed, certain metas and tactics are likely more meaningful at the time; however, once playstyles evolve at later record dates, certain other stats become more important which the already trained model does not know. The temporal nature of this problem seems to be more of a challenge than anticipated.